# 拓展活动：寻找最优模型，并保存

In [1]:
import numpy as np  #数据库
import keras
import tensorflow as tf
#从keras库导入处理神经网络模型model所需的函数。
from keras.models import Sequential   #序贯模型
from keras.models import load_model 
#从keras库导入设计神经网络模型的各个层次layers需要用到的函数。
from keras.layers import Dense,Dropout,Activation,Flatten,GlobalAveragePooling2D  #构建 keras网络结构
from keras.layers import Conv2D, MaxPooling2D
#从keras库导入神经网络模型的优化器
from keras.optimizers import SGD   #SGD随机梯度下降

%matplotlib inline
import cv2
import matplotlib.pyplot as plt    #画图

Using TensorFlow backend.
C:\Users\67897\Anaconda3\lib\site-packages\tensorflow\python\framework\dtypes.py:526: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.
  _np_qint8 = np.dtype([("qint8", np.int8, 1)])
C:\Users\67897\Anaconda3\lib\site-packages\tensorflow\python\framework\dtypes.py:527: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.
  _np_quint8 = np.dtype([("quint8", np.uint8, 1)])
C:\Users\67897\Anaconda3\lib\site-packages\tensorflow\python\framework\dtypes.py:528: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.
  _np_qint16 = np.dtype([("qint16", np.int16, 1)])
C:\Users\67897\Anaconda3\lib\site-packages\tensorflow\python\framework\dtypes.py

In [2]:
#下面的函数get_nb_files（）功能是：获取指定目录directory下面的所有文件的总个数
import os
import glob
def get_nb_files(directory):
  """Get number of files by searching directory recursively"""
  if not os.path.exists(directory):
    return 0
  cnt = 0
  for r, dirs, files in os.walk(directory):
    for dr in dirs:
      cnt += len(glob.glob(os.path.join(r, dr + "/*")))
  return cnt

In [3]:
train_dir = 'mnist//train'
val_dir = 'mnist//test'
nb_train_samples = get_nb_files(train_dir)      # 训练样本个数
nb_val_samples = get_nb_files(val_dir)       #测试集样本个数
 
    
print('用于训练的样本总数nb_train_samples:',nb_train_samples,' 用于测试检验的样本总数nb_val_samples:',nb_val_samples)

#模型训练的参数,，跟输入图像的分类数目有关，跟图像的尺寸也有关。
# nb_classes= 10 #分类数，此行可以省略，因为下面一行会根据子文件夹的个数自动计算分类数。
nb_classes = len(glob.glob(train_dir + "/*"))   #我的分类总数，就是训练文件夹train和测试文件夹test下面的子文件夹的个数
print("分类数nb_classes:",nb_classes)
IM_WIDTH, IM_HEIGHT = 299, 299  #InceptionV3指定的图片尺寸为299,299
FC_SIZE = 1024                # 全连接层的节点个数
NB_IV3_LAYERS_TO_FREEZE = 311  # 冻结层的数量 xjj change to 200

# nb_epoch = 200
nb_epoch = 10 #我的训练次数，调试代码阶段，将其设置为1或小于5的数
batch_size = 512 #batch_size一般设置为总文件数的1%左右，如果太小，训练速度会很慢。
lrate=0.001#设置为0.1，0.001或0.0001，需要选择，越小，则训练耗时越长，但准确率未必越高

用于训练的样本总数nb_train_samples: 60000  用于测试检验的样本总数nb_val_samples: 10000
分类数nb_classes: 10


In [4]:
#数据准备：使用图片生成器ImageDataGenerator从原始数据生成更多数据
#　图片生成器ImageDataGenerator,通过拉伸，平移，旋转等使得数据量增大
from keras.preprocessing.image import ImageDataGenerator
from keras.applications.inception_v3 import InceptionV3, preprocess_input
from keras.models import Model
from keras.optimizers import SGD
from keras.optimizers import Adam
train_datagen = ImageDataGenerator(
     rescale = 1./255,#!!!此行最好不省略。
     preprocessing_function=preprocess_input)  #下列4行的数据变换是否有，基本不会增加训练耗时
#      rotation_range=20,  #图片随机转动的角度
#      width_shift_range=0.2, #图片随机水平偏移的幅度
#      height_shift_range=0.2, #图片随机竖直偏移的幅度
#      fill_mode='nearest',  #当进行变换时超出边界的点将根据本参数给定的方法进行处理
#      horizontal_flip=False) #适用于水平翻转不影响图片语义的时候

test_datagen = ImageDataGenerator(
     rescale = 1./255,#!!!此行最好不省略。
     preprocessing_function=preprocess_input)  

# 从所给的原始的文件夹 生成训练数据train_generator与测试数据validation_generator
train_generator = train_datagen.flow_from_directory(
train_dir,  #原始的train文件夹
target_size=(IM_WIDTH, IM_HEIGHT),
batch_size=batch_size,
class_mode='categorical')
 
validation_generator = test_datagen.flow_from_directory(
val_dir,    #原始的test文件夹
target_size=(IM_WIDTH, IM_HEIGHT),
batch_size=batch_size,
class_mode='categorical')

Found 60000 images belonging to 10 classes.
Found 10000 images belonging to 10 classes.


In [1]:
#mnist的模型设计和训练

#从谷歌设计的模型inception v3出发，调整模型。调整之前先准备好工具：add_new_last_layer（）和setup_to_finetune()
# 下列函数add_new_last_layer（）的功能是：输入的base_model是inceptionV3模型，
#将最后一层的分类数改为自己问题的分类总数nb_classes，然后返回设计好的新的模型
def add_new_last_layer(base_model, nb_classes):
  x = base_model.output
  x = GlobalAveragePooling2D()(x)
  x = Dense(FC_SIZE, activation='relu')(x) #全连接层FC layer, random init
  predictions = Dense(nb_classes, activation='softmax')(x) #将最后一层的分类数改为自己问题的分类总数nb_classes
  model = Model(input=base_model.input, output=predictions)
  return model

# 设置模型的权值是否可修改，冻结NB_IV3_LAYERS之前的层
#[0,NB_IV3_LAYERS_TO_FREEZE)层的权值都不可修改，[NB_IV3_LAYERS_TO_FREEZE,最后一层]的权值都需要重新学习
def setup_to_finetune(model):
  for layer in model.layers[:NB_IV3_LAYERS_TO_FREEZE]: #[0,NB_IV3_LAYERS_TO_FREEZE)层的权值都不可修改
     layer.trainable = False
  for layer in model.layers[NB_IV3_LAYERS_TO_FREEZE:]:#[NB_IV3_LAYERS_TO_FREEZE,最后一层]的权值都需要重新学习
     layer.trainable = True
  model.compile(optimizer=Adam(lr=lrate), loss='categorical_crossentropy', metrics=['accuracy'])

 # 开始使用工具设置神经网络结构
model = InceptionV3(weights='imagenet', include_top=False) #从inceptionV3模型出发，不使用它的top层的权值
model = add_new_last_layer(model, nb_classes)
#model.summary()
setup_to_finetune(model)
model.summary()
# model.layers[NB_IV3_LAYERS_TO_FREEZE:]#查看一下NB_IV3_LAYERS_TO_FREEZE（默认设为311）后面有几层，

#优化器选择方法参考 https://keras.io/zh/optimizers/
#编译模型方案1，优化器选sgd
# sgd = SGD(lr=0.01, decay=1e-6, momentum=0.9, nesterov=True) # 优化函数，设定学习率（lr）等参数
#编译模型方案2，优化器选Adadelta
# adadelta = keras.optimizers.Adadelta(lr=1.0, rho=0.95, epsilon=None, decay=0.0)
#编译模型方案3，优化器选RMSprop
# rmsprop = keras.optimizers.RMSprop(lr=0.001, rho=0.9, epsilon=None, decay=0.0)
#编译模型方案4，优化器选Adam
# adam = keras.optimizers.Adam(lr=0.001, beta_1=0.9, beta_2=0.999, epsilon=None, decay=0.0, amsgrad=False)

sgd = SGD(lr=0.01, decay=1e-6, momentum=0.9, nesterov=True) # 优化函数，设定学习率（lr）等参数
model.compile(optimizer=sgd,loss='categorical_crossentropy', metrics=['accuracy']) 
print("数据导入和预处理已经完成，模型设计完毕")

NameError: name 'InceptionV3' is not defined

In [ ]:
#开始训练模型fit
import time 
t1 = time.strftime("%Y/%m/%d  %H:%M:%S")
print("模型训练开始时间starting time:",t1) ##24小时格式 
# 模型训练
history_ft = model.fit_generator(
train_generator,
steps_per_epoch=nb_train_samples/batch_size,
nb_epoch=nb_epoch,
validation_data=validation_generator,
validation_steps=nb_val_samples/batch_size,
class_weight='auto1',verbose=1) #verbose=2的意思：每个epoch没有时间戳？

模型训练开始时间starting time: 2021/12/06  18:54:20
Instructions for updating:
Use tf.cast instead.


C:\Users\67897\Anaconda3\lib\site-packages\ipykernel_launcher.py:12: UserWarning: The semantics of the Keras 2 argument `steps_per_epoch` is not the same as the Keras 1 argument `samples_per_epoch`. `steps_per_epoch` is the number of batches to draw from the generator at each epoch. Basically steps_per_epoch = samples_per_epoch/batch_size. Similarly `nb_val_samples`->`validation_steps` and `val_samples`->`steps` arguments have changed. Update your method calls accordingly.
  if sys.path[0] == '':
C:\Users\67897\Anaconda3\lib\site-packages\ipykernel_launcher.py:12: UserWarning: Update your `fit_generator` call to the Keras 2 API: `fit_generator(<keras_pre..., steps_per_epoch=117.1875, validation_data=<keras_pre..., validation_steps=19.53125, class_weight="auto1", verbose=1, epochs=1)`
  if sys.path[0] == '':


Epoch 1/1
  5/117 [>.............................] - ETA: 2:57:27 - loss: 2.3378 - acc: 0.1148

In [ ]:
import time
t2 = time.strftime("%Y/%m/%d  %H:%M:%S")
print("模型训练开始和结束时间分别为 ：", t1, t2)
# 模型保存
# model.save("newModel1.h5",t2) 

In [ ]:
#acc
import matplotlib.pyplot as plt
plt.plot(history_ft.history['acc'])
plt.plot(history_ft.history['val_acc'])
plt.title('model accuracy')
plt.ylabel('accuracy')
plt.xlabel('epoch')
plt.legend(['train', 'test'], loc='lower right')
plt.savefig('acc.jpg')

In [ ]:
#loss
plt.plot(history_ft.history['loss'])
plt.plot(history_ft.history['val_loss'])
plt.title('model loss')
plt.ylabel('loss')
plt.xlabel('epoch')
plt.legend(['train', 'test'], loc='upper right')
plt.savefig('loss.jpg')

In [ ]:
e